# Use Case 1 — Vulnerable E-commerce Support Chatbot

## 
Before protecting a GenAI application, participants should understand the **baseline application flow**.

This notebook intentionally builds a very small customer-support chatbot with minimal security controls.

## Business Scenario
A retailer wants an AI assistant that answers:
- order questions
- delivery questions
- refund questions
- return questions
- product questions

## expectations
1. How a Python application calls an LLM.
2. Where user input enters the system.
3. Why natural-language input must be treated as untrusted.
4. Why model output should not automatically be trusted.
5. Why we need a baseline before testing security controls.

## Architecture

```text
+-----------+      +----------------------+      +-------------+      +-----------+
| Customer  | ---> | Python Support App   | ---> | OpenAI LLM  | ---> | Response  |
+-----------+      +----------------------+      +-------------+      +-----------+
                         |
                         +-- No input scoping
                         +-- No prompt-injection detector
                         +-- No output-security validation
```

## Security Message
This is a ** vulnerable architecture**.

We first observe how the system behaves normally.  
Later notebooks attack it and then add first-line controls.

In [1]:
# Install required libraries once before running the notebook.
# pip install openai pandas python-dotenv

## Step 1 — Configure the Environment

Create a `.env` file beside the notebook:

```text
OPENAI_API_KEY=your_api_key_here
OPENAI_MODEL=gpt-5.5
```

The model name is configurable so you can change it without editing every code cell.

In [2]:
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")
API_KEY = os.getenv("OPENAI_API_KEY")

print("Model configured:", MODEL)
print("API key available:", bool(API_KEY))

client = OpenAI(api_key=API_KEY)

Model configured: gpt-5.5
API key available: True


## Step 2 — Define the Business Scope

A support assistant should have a clear role.

At this stage the instruction is intentionally simple so we can later compare it with a hardened version.

In [3]:
SYSTEM_PROMPT = '''
You are an e-commerce customer-support assistant.
Answer questions about orders, deliveries, returns, refunds, and products.
Be concise and helpful.
'''

print(SYSTEM_PROMPT)


You are an e-commerce customer-support assistant.
Answer questions about orders, deliveries, returns, refunds, and products.
Be concise and helpful.



## Step 3 — Create a Small LLM Function

The function accepts one user prompt and sends it to the model.

Notice that:
- the user prompt is not validated,
- no attack detection happens,
- the output is returned directly.

In [4]:
def call_support_bot(user_prompt: str) -> str:
    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=user_prompt
    )
    return response.output_text

## Step 4 — Run One Normal Prompt

Always begin with a normal request.

This gives us the **baseline behavior**.

In [5]:
prompt = "What is your return policy for electronics?"
response = call_support_bot(prompt)

print("USER:")
print(prompt)
print("\nASSISTANT:")
print(response)

USER:
What is your return policy for electronics?

ASSISTANT:
Electronics can typically be returned within **30 days of delivery** if they are:

- In **original condition**
- Returned with **all accessories, manuals, and packaging**
- Not damaged, misused, or missing parts
- **Factory reset** with any personal data removed

Some items, such as opened software, downloadable products, gift cards, or certain hygiene-related electronics, may be non-returnable. Defective electronics may qualify for a replacement, refund, or warranty support.

If you share the product or order number, I can check the exact return eligibility.


## Step 5 — Run Several Normal Support Requests

This checks whether the application is useful before we test attacks.

In [6]:
normal_prompts = [
    "Where is order ORD-25001?",
    "How long does a refund normally take?",
    "My laptop arrived damaged. What should I do?",
    "Can I change my delivery address?"
]

for prompt in normal_prompts:
    print("\n" + "="*80)
    print("USER:", prompt)
    print("ASSISTANT:", call_support_bot(prompt))


USER: Where is order ORD-25001?
ASSISTANT: I can help check that, but I don’t have live order lookup access here.

Please provide the email address or phone number used for order **ORD-25001** and the delivery postcode/ZIP, and I can help you verify the status or guide you on the next steps.

USER: How long does a refund normally take?
ASSISTANT: Refunds normally take **5–10 business days** to appear in your account after they’re processed, depending on your bank or payment provider.

If your refund was issued to a card, your bank may take a few extra days to post it.

USER: My laptop arrived damaged. What should I do?
ASSISTANT: I’m sorry your laptop arrived damaged. Here’s what to do:

1. **Take photos/videos** of the damage, the shipping box, packaging, and the laptop from multiple angles.  
2. **Keep all packaging** and accessories—don’t throw anything away yet.  
3. **Stop using the laptop** if there’s any visible damage or safety concern.  
4. **Contact customer support as soon 

## Step 6 — Identify the Trust Boundaries

| Boundary | Why it matters |
|---|---|
| Customer → App | User-controlled text is untrusted |
| App → LLM | The app decides what instructions/context the model receives |
| LLM → App | Generated output is probabilistic |
| App → User/Other System | Unsafe output may create business impact |



## Step 7 — Record the Baseline

A useful security lab always records:
- what the system was supposed to do,
- what it actually did,
- what changed after security controls were added.

In the next notebook we will create repeatable attack tests from CSV.

## Expected Outcome



```text
The LLM is not the entire application.
```

Security must protect:
- inputs,
- instructions,
- context,
- output,
- downstream actions.